In [1]:
# basic
import os
import pickle
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
# pre processing
from sklearn import preprocessing as pre
# NN
import torch
import torch.nn as nn
from torch import Tensor
import torch.nn.functional as F
import torch.optim as optim
from torch.nn import MSELoss
from torch_geometric.nn import GCNConv
# val and plot
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error
from loguru import logger as log
#from ..val import calculate_metrics
# plot
import matplotlib.pyplot as plt
# foundation model
from functools import reduce

/home/marcos/.pyenv/versions/3.10.13/envs/gnn-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import itertools
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [3]:
import pmdarima as pm

In [4]:
plt.style.use("seaborn-v0_8-whitegrid")

In [5]:
SEED = 1345
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
seed_everything(SEED)
#plt.style.use('seaborn-whitegrid')
#pd.set_option('display.float_format', '{:.16f}'.format)
warnings.filterwarnings('ignore')

In [6]:
def load_datasets(filepath):
    """Carrega os datasets de arquivos pickle."""
    try:
        with open(filepath, 'rb') as f:
            dataset = pickle.load(f)
        return dataset
    except IOError as e:
        log.error(f"Erro ao carregar o dataset: {e}")
    except pickle.PickleError as e:
        log.error(f"Erro ao desserializar o dataset: {e}")
        traceback.print_exception(e)

## Data

In [7]:
sb = pd.read_parquet("/home/marcos/loader_03-04_2024.parquet")
sb.head()

,125960550,230565994,258781031,43768720,44072192,44783654,44783914,44784438,45833547,47568123
2024-03-01 05:00:00,1.333333,0.000000,70.792221,41.188599,1.724359,14.955100,2.032506,3.571820,5.877792,7.546274
2024-03-01 05:30:00,3.595238,1.583333,229.051071,172.071198,8.282966,44.559937,11.048912,18.211931,18.912033,18.276293
2024-03-01 06:00:00,4.812975,3.268518,424.853729,433.062469,18.825665,97.263435,26.276600,41.471294,40.731876,37.141144
2024-03-01 06:30:00,9.215629,5.256614,630.444153,743.177368,25.593414,149.329544,49.763138,71.520836,57.200085,53.487366
2024-03-01 07:00:00,12.585028,6.152447,841.874512,1132.739502,44.350349,204.275940,78.721497,107.241295,77.808769,75.446609


In [8]:
sb.columns

Index(['125960550', '230565994', '258781031', '43768720', '44072192',
       '44783654', '44783914', '44784438', '45833547', '47568123'],
      dtype='object')

In [9]:
# define X and Y
sbx = sb.query("index <= '2024-03-31 23:59:59'")
sbx.shape, sb.shape

((1238, 10), (3640, 10))

In [10]:
# define X and Y
sby = sb.query("index > '2024-03-31 23:59:59'")
sby.shape, sb.shape

((2402, 10), (3640, 10))

In [11]:
pred_len = abs(sbx.shape[0] - sb.shape[0])
pred_len

2402

In [12]:
sby.shape

(2402, 10)

In [13]:
## Grid Search

In [14]:
node = ["125960550"]
serie = sbx[node]

In [15]:
# Auto ARIMA com busca de parâmetros, incluindo sazonalidade
modelo_auto = pm.auto_arima(
    serie,
    start_p=0, max_p=3,
    start_q=0, max_q=3,
    d=None,            # auto determina o melhor d
    seasonal=True,
    start_P=0, max_P=2,
    start_Q=0, max_Q=2,
    D=None,            # auto determina o melhor D
    m=24,              # frequência sazonal, ex: 24 para dados horários
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True      # mais rápido
)

print("Resultados pos grid search")
print(modelo_auto.summary())

Performing stepwise search to minimize aic
 ARIMA(0,0,0)(0,0,0)[24] intercept   : AIC=8008.676, Time=0.02 sec
 ARIMA(1,0,0)(1,0,0)[24] intercept   : AIC=4499.395, Time=1.02 sec
 ARIMA(0,0,1)(0,0,1)[24] intercept   : AIC=6548.585, Time=0.87 sec
 ARIMA(0,0,0)(0,0,0)[24]             : AIC=10015.926, Time=0.01 sec
 ARIMA(1,0,0)(0,0,0)[24] intercept   : AIC=4499.158, Time=0.07 sec
 ARIMA(1,0,0)(0,0,1)[24] intercept   : AIC=4498.710, Time=1.02 sec
 ARIMA(1,0,0)(1,0,1)[24] intercept   : AIC=4501.611, Time=2.12 sec
 ARIMA(1,0,0)(0,0,2)[24] intercept   : AIC=4476.307, Time=3.26 sec
 ARIMA(1,0,0)(1,0,2)[24] intercept   : AIC=4478.057, Time=8.27 sec
 ARIMA(0,0,0)(0,0,2)[24] intercept   : AIC=7976.507, Time=2.63 sec
 ARIMA(2,0,0)(0,0,2)[24] intercept   : AIC=4352.438, Time=3.20 sec
 ARIMA(2,0,0)(0,0,1)[24] intercept   : AIC=4351.615, Time=1.16 sec
 ARIMA(2,0,0)(0,0,0)[24] intercept   : AIC=4360.555, Time=0.09 sec
 ARIMA(2,0,0)(1,0,1)[24] intercept   : AIC=inf, Time=3.24 sec
 ARIMA(2,0,0)(1,0,0)[24

In [16]:
with open(f"sarima_fit_gs/{node}-modelo_auto_sarima.pkl", "wb") as f:
    pickle.dump(modelo_auto, f)

## Data in batch

In [17]:
fpath_root = "/mnt/data/marcos/data/node_regression_bus/data_split/"
test_dataset = load_datasets(f'{fpath_root}test.pkl')
# x (dados de input)
test_dataset[0].x.shape

torch.Size([2871, 280])

In [18]:
inference_size = test_dataset[0].y.shape[1]
inference_size

200

In [19]:
nodes = [53,  365,  382,  666,  701, 1326, 1404, 1569, 1916, 2617]

In [20]:
stops = {53: '125960550',
         365: '230565994',
         382: '258781031',
         666: '43768720',
         701: '44072192',
         1326: '44783654',
         1404: '44783914',
         1569: '44784438',
         1916: '45833547',
         2617: '47568123'}

In [21]:
modelo_auto

ARIMA(order=(2, 0, 3), scoring_args={}, seasonal_order=(0, 0, 1, 24),
      suppress_warnings=True)

In [22]:
modelo_auto.order

(2, 0, 3)

In [23]:
modelo_auto.seasonal_order

(0, 0, 1, 24)

In [24]:
serie_np = serie[stops[nodes[0]]].values  # array tipo numpy
serie_np.shape

(1238,)

In [25]:
nova_seq = test_dataset[0].x[nodes[0], :].cpu().numpy()
nova_seq.shape

(280,)

In [26]:
280 /  40

7.0

In [27]:
# Concatenação simples das duas sequências
serie_expandida = np.concatenate([serie_np, nova_seq])
serie_expandida

array([1.3333334, 3.5952382, 4.812975 , ..., 6.357711 , 4.9194756,
       4.5028086], dtype=float32)

### Forecasting

In [ ]:
scores_error = {'node': [], 'batch': [], 'mae': [], 'mse': [], 'r2': [], 'mape': []}
targets = []
cost, time = 0, 0
dfs = []

only_day = False

for node in nodes:
    dfs_pred = []    
    for time, snapshot in tqdm(enumerate(test_dataset)):
        snapshot.to('cpu')
        
        serie = sbx.copy()#[stops[node]]

        #
        # Data
        #
        if only_day:
            serie_np = serie[stops[node]].values  # array tipo numpy
            nova_seq = snapshot.x[node, :].cpu().numpy()
            serie_expandida = np.concatenate([serie_np, nova_seq[-40:]])
        else:
            only_day = True
            serie_np = serie[stops[node]].values  # array tipo numpy
            nova_seq = snapshot.x[node, :].cpu().numpy()
            serie_expandida = np.concatenate([serie_np, nova_seq])
        
        #
        # (alterar aqui o modelo)
        #
        
        # ler os parametros do no
        with open(f"sarima_fit_gs/{stops[node]}-grid-search-results.pkl", "rb") as f:
            modelo_auto = pickle.load(f)
        
        modelo = SARIMAX(serie_expandida,
                         order=modelo_auto.order,
                         seasonal_order=modelo_auto.seasonal_order,
                         enforce_stationarity=False,
                         enforce_invertibility=False)

        resultado = modelo.fit()
        #
        y_true = snapshot.y[node, :].cpu().numpy()
        forecast = resultado.get_forecast(steps=y_true.shape[0])
        y_pred = forecast.predicted_mean
        
        scores_error['node'].append(stops[node])
        scores_error['batch'].append(time)
        scores_error['mse'].append(mean_squared_error(y_true, y_pred))
        scores_error['mae'].append(mean_absolute_error(y_true, y_pred))
        scores_error['r2'].append(r2_score(y_true, y_pred))
        scores_error['mape'].append(mean_absolute_percentage_error(y_true, y_pred))
        
        targets.append({'true': y_true,
                        'pred': y_pred})
    

38it [00:55,  1.46s/it]
38it [09:11, 14.53s/it]
38it [00:10,  3.73it/s]
32it [10:26, 20.45s/it]

In [ ]:
df_results = pd.DataFrame(scores_error)
df_results